# Part 4g: Multi-Symbol LOB Manifold — Unified Representation, DR Comparison & Regime Detection

*"If the order book has structure in one stock, does it have the same structure in all stocks?"*

Part 4f showed that NVDA's 10-level OBI vector lives on a 2D manifold, with ISOMAP recovering 96% of its structure. This notebook asks the harder question: **is that structure consistent across different symbols, and can we build a single unified representation that spans them?**

**Extensions over Part 4f:**
1. **Larger dataset** — full month of MBP-10 data (Oct 2023) via Databento API
2. **Multi-symbol pooling** — fit one DR model on NVDA + AAPL + TSLA + MSFT + SPY simultaneously; mixing symbols in the embedding
3. **Unified representation** — within-symbol z-score normalization so all stocks are comparable on the same embedding
4. **DR method comparison** — ISOMAP, t-SNE, UMAP, PCA side-by-side with quantitative metrics
5. **Regime clustering** — K-Means on the unified embedding to detect market micro-regimes across all symbols
6. **Calm vs stress** — project the Aug 2024 BOJ shock period onto the calm manifold

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import spearmanr
from dotenv import load_dotenv

# import databento as db

from sklearn.manifold import Isomap, TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, pairwise_distances
from sklearn.manifold import trustworthiness
import umap

warnings.filterwarnings('ignore')

load_dotenv()
API_KEY = os.environ.get('DATABENTO_API_KEY', '')
if not API_KEY:
    raise ValueError('Set DATABENTO_API_KEY in your .env file — do NOT hardcode keys here.')
w
LOB_DIR   = 'data/lob'
FIG_DIR   = 'figures'
os.makedirs(FIG_DIR, exist_ok=True)

SYMBOLS   = ['NVDA', 'AAPL', 'TSLA', 'MSFT', 'SPY']
DATASET   = 'XNAS.ITCH'
OBI_COLS  = [f'obi_{k:02d}' for k in range(10)]

SYM_COLORS = {
    'NVDA': '#E74C3C',
    'AAPL': '#4472C4',
    'TSLA': '#2ECC71',
    'MSFT': '#F39C12',
    'SPY' : '#9B59B6',
}

## Section 1: Data Download via Databento API

We download **MBP-10** (10-level order book) data for 5 symbols across two periods:

| Period | Dates | Days | Description |
|---|---|---|---|
| Calm | Oct 2 – Oct 31, 2023 | ~22 trading days | Normal regime — baseline manifold |
| Stress | Aug 5 – Aug 9, 2024 | 4 trading days | BOJ shock — test regime detection |

Files are cached to `data/lob/` and skipped if they already exist.

In [ ]:
DOWNLOAD_PERIODS = {
    'calm_oct2023'   : ('2023-10-02', '2023-11-01'),
    'stress_aug2024' : ('2024-08-05', '2024-08-10'),
}

client = db.Historical(API_KEY)

for period_name, (start, end) in DOWNLOAD_PERIODS.items():
    for sym in SYMBOLS:
        out_path = f'{LOB_DIR}/lob_mbp10_{sym}_{period_name}_full.parquet'
        if os.path.exists(out_path):
            df_ex = pd.read_parquet(out_path, columns=['bid_px_00'])
            print(f'  [skip] {sym} {period_name}: {len(df_ex):,} rows already on disk')
            continue

        print(f'Downloading mbp-10 | {sym} | {period_name} | {start} -> {end} ...')
        try:
            data = client.timeseries.get_range(
                dataset  = DATASET,
                schema   = 'mbp-10',
                symbols  = [sym],
                start    = start,
                end      = end,
                stype_in = 'raw_symbol',
            )
            df = data.to_df()
            df.to_parquet(out_path)
            print(f'  Saved {len(df):,} rows -> {out_path}')
        except Exception as e:
            print(f'  ERROR {sym} {period_name}: {e}')

print('\nDownload complete.')

## Section 2: Feature Engineering — 1-Minute OBI Bars

For each symbol, we:
1. Load the MBP-10 parquet and filter to RTH (09:30–16:00 ET)
2. Compute `OBI_k = (bid_sz_k − ask_sz_k) / (bid_sz_k + ask_sz_k)` for levels 0–9
3. Resample to **1-minute bars** (mean OBI, last mid-price)
4. Tag each bar with its symbol

**Why 1-minute?** It strikes the right balance: fine enough to capture intraday structure, coarse enough that the mean OBI is stable and not dominated by single-tick noise. At tick resolution the OBI jumps discontinuously; at 5-minute bars you lose too much intraday dynamics.

In [ ]:
def build_obi_bars(sym, period_name):
    """Load mbp-10, filter RTH, build 1-min OBI bars. Returns DataFrame."""
    path = f'{LOB_DIR}/lob_mbp10_{sym}_{period_name}_full.parquet'
    df   = pd.read_parquet(path)
    df.index = pd.DatetimeIndex(df.index).tz_convert('America/New_York')
    mh = df.between_time('09:30', '16:00')

    frames = {}
    for k in range(10):
        b = mh[f'bid_sz_{k:02d}'].astype(np.int64)
        a = mh[f'ask_sz_{k:02d}'].astype(np.int64)
        denom = (b + a).replace(0, np.nan)
        frames[f'obi_{k:02d}'] = ((b - a) / denom).resample('1min').mean()
    frames['mid'] = ((mh['bid_px_00'] + mh['ask_px_00']) / 2).resample('1min').last()

    out = pd.DataFrame(frames).dropna()
    out['ret_fwd'] = out['mid'].pct_change().shift(-1)
    out['symbol']  = sym
    return out.dropna()


calm_dfs = {}
for sym in SYMBOLS:
    calm_dfs[sym] = build_obi_bars(sym, 'calm_oct2023')
    d = calm_dfs[sym]
    print(f'{sym}: {len(d):,} bars | {d.index.min().date()} -> {d.index.max().date()}')

print(f'\nTotal calm bars: {sum(len(v) for v in calm_dfs.values()):,}')

## Section 3: Cross-Symbol OBI Structure

Before building a unified embedding, we check whether the OBI correlation structure is **consistent across symbols**. If each symbol has a similar banded correlation pattern (adjacent levels correlated, far levels independent), that motivates a single shared manifold. If structures differ, a symbol-specific manifold might be needed.

The key question: is the L0 vs L9 independence universal, or NVDA-specific?

In [ ]:
fig, axes = plt.subplots(1, len(SYMBOLS), figsize=(18, 3.5))

l0_l9_corrs = {}
for ax, sym in zip(axes, SYMBOLS):
    X = calm_dfs[sym][OBI_COLS].values
    corr = pd.DataFrame(X, columns=OBI_COLS).corr().values
    im = ax.imshow(corr, cmap='RdBu', vmin=-1, vmax=1)
    ax.set_xticks(range(10))
    ax.set_xticklabels([f'L{k}' for k in range(10)], fontsize=6)
    ax.set_yticks(range(10))
    ax.set_yticklabels([f'L{k}' for k in range(10)], fontsize=6)
    ax.set_title(f'{sym}', fontsize=10, color=SYM_COLORS[sym], fontweight='bold')
    for i in range(10):
        for j in range(10):
            ax.text(j, i, f'{corr[i,j]:.1f}', ha='center', va='center',
                    fontsize=4.5, color='white' if abs(corr[i,j]) > 0.6 else 'black')
    l0_l9_corrs[sym] = corr[0, 9]

plt.colorbar(im, ax=axes[-1], shrink=0.9)
plt.suptitle('OBI Cross-Level Correlation Heatmaps — All 5 Symbols (Oct 2023, 1-min bars)',
             fontsize=11, y=1.03)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/4g_obi_corr_all_symbols.png', dpi=150, bbox_inches='tight')
plt.show()

print('L0 vs L9 correlation (measures top-of-book vs deep-book independence):')
for sym, c in l0_l9_corrs.items():
    print(f'  {sym}: {c:+.3f}')

## Section 4: Unified Representation — Cross-Symbol Normalization

**The problem with raw OBI pooling:** NVDA's OBI swings are not the same magnitude as SPY's. Directly concatenating raw OBI vectors would let the model confuse "which stock" with "what regime."

**Solution — within-symbol z-score normalization:**

$$\tilde{\mathbf{x}}^{(s)}_t = \frac{\mathbf{x}^{(s)}_t - \boldsymbol{\mu}^{(s)}}{\boldsymbol{\sigma}^{(s)}}$$

where $\boldsymbol{\mu}^{(s)}$ and $\boldsymbol{\sigma}^{(s)}$ are the per-symbol mean and standard deviation computed on the **training** split only (first 15 trading days). This maps each symbol's OBI into a common scale so the manifold captures *book shape* rather than *level differences*.

After normalization, we pool all symbols into one matrix and hold out the last 7 trading days as OOS.

In [ ]:
TRAIN_CUTOFF = '2023-10-23'

scalers      = {}
train_blocks = []
oos_blocks   = []

for sym in SYMBOLS:
    df   = calm_dfs[sym]
    mask_train = df.index.date <= pd.Timestamp(TRAIN_CUTOFF).date()
    mask_oos   = df.index.date >  pd.Timestamp(TRAIN_CUTOFF).date()

    scaler = StandardScaler()
    X_tr   = scaler.fit_transform(df.loc[mask_train, OBI_COLS].values)
    X_oos  = scaler.transform(df.loc[mask_oos,   OBI_COLS].values)

    scalers[sym] = scaler

    tr_meta  = df.loc[mask_train, ['ret_fwd']].copy()
    oos_meta = df.loc[mask_oos,   ['ret_fwd']].copy()
    tr_meta['symbol']  = sym
    oos_meta['symbol'] = sym

    train_blocks.append((X_tr,  tr_meta))
    oos_blocks.append  ((X_oos, oos_meta))
    print(f'{sym}  train: {len(X_tr):,}  |  oos: {len(X_oos):,}')

X_pool_train = np.vstack([b[0] for b in train_blocks])
X_pool_oos   = np.vstack([b[0] for b in oos_blocks])
meta_train   = pd.concat([b[1] for b in train_blocks])
meta_oos     = pd.concat([b[1] for b in oos_blocks])

sym_train = meta_train['symbol'].values
sym_oos   = meta_oos['symbol'].values
ret_train = meta_train['ret_fwd'].values
ret_oos   = meta_oos['ret_fwd'].values

print(f'\nPooled train: {X_pool_train.shape}  |  OOS: {X_pool_oos.shape}')

## Section 5: Fitting Four DR Methods on the Pooled Multi-Symbol Data

We fit four 2D embeddings on the same pooled (normalized) OBI matrix:

| Method | Type | Preserves | Hyperparameters |
|---|---|---|---|
| **PCA** | Linear | Global variance | — |
| **ISOMAP** | Non-linear (geodesic) | Global manifold structure | `n_neighbors=15` |
| **t-SNE** | Non-linear (KL divergence) | Local neighbourhood | `perplexity=50` |
| **UMAP** | Non-linear (topological) | Local + global | `n_neighbors=30`, `min_dist=0.1` |

**Note on t-SNE:** t-SNE embeddings are not directly out-of-sample invertible (no `.transform()`). For OOS we project using ISOMAP and UMAP's parametric extension.

In [ ]:
print('Fitting PCA...')
pca   = PCA(n_components=2, random_state=42)
Z_pca = pca.fit_transform(X_pool_train)
print(f'  PCA variance explained: {pca.explained_variance_ratio_.sum()*100:.1f}%')

print('Fitting ISOMAP...')
iso   = Isomap(n_components=2, n_neighbors=15)
Z_iso = iso.fit_transform(X_pool_train)
print(f'  ISOMAP reconstruction error: {iso.reconstruction_error():.4f}')

print('Fitting t-SNE...')
tsne  = TSNE(n_components=2, perplexity=50, random_state=42, n_iter=1000)
Z_tsne = tsne.fit_transform(X_pool_train)
print(f'  t-SNE KL divergence: {tsne.kl_divergence_:.4f}')

print('Fitting UMAP...')
umap_model = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.1, random_state=42)
Z_umap = umap_model.fit_transform(X_pool_train)
print('  UMAP done.')

Z_pool_oos_iso  = iso.transform(X_pool_oos)
Z_pool_oos_umap = umap_model.transform(X_pool_oos)
Z_pool_oos_pca  = pca.transform(X_pool_oos)

EMBEDDINGS = {
    'PCA'   : Z_pca,
    'ISOMAP': Z_iso,
    't-SNE' : Z_tsne,
    'UMAP'  : Z_umap,
}

## Section 6: Visualising the Unified Embedding

Each plot shows the same pooled data from a different algorithm's perspective, colored by symbol. Key questions:
- Do symbols **cluster separately** (each method sees them as distinct) or **intermix** (symbols share a common structure)?
- Does any algorithm reveal structure that others miss?
- Is the embedding compact (well-separated clusters) or diffuse (gradients)?

If symbols **intermix**, the OBI manifold is universal — the same book dynamics appear regardless of which stock. If they **separate**, each symbol has idiosyncratic dynamics.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 13))
axes = axes.flatten()

for ax, (name, Z) in zip(axes, EMBEDDINGS.items()):
    for sym in SYMBOLS:
        mask = sym_train == sym
        ax.scatter(Z[mask, 0], Z[mask, 1], c=SYM_COLORS[sym], s=6, alpha=0.35, label=sym)
    ax.set_title(f'{name}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Component 1')
    ax.set_ylabel('Component 2')
    ax.legend(fontsize=8, markerscale=2)

plt.suptitle('Unified 2D Embedding — 5 Symbols, Pooled & Z-Score Normalized (Oct 2023)\n'
             'Color = symbol; overlap means shared manifold structure',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/4g_dr_comparison_by_symbol.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
tod_train = pd.DatetimeIndex(meta_train.index).hour + pd.DatetimeIndex(meta_train.index).minute / 60
vmax_ret  = np.percentile(np.abs(ret_train), 95)

fig, axes = plt.subplots(2, 2, figsize=(16, 13))
axes = axes.flatten()

for ax, (name, Z) in zip(axes, EMBEDDINGS.items()):
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=tod_train, cmap='plasma',
                    s=5, alpha=0.3, vmin=9.5, vmax=16)
    plt.colorbar(sc, ax=ax, label='Hour ET')
    ax.set_title(f'{name} — colored by Time of Day', fontsize=11, fontweight='bold')
    ax.set_xlabel('Component 1')
    ax.set_ylabel('Component 2')

plt.suptitle('Unified 2D Embedding — Colored by Time of Day (all 5 symbols)',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/4g_dr_comparison_by_tod.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 7: Quantitative DR Method Comparison

We evaluate each embedding using three metrics:

| Metric | What it measures | Better when |
|---|---|---|
| **Trustworthiness** | Are each point's 2D neighbours also its high-D neighbours? | Closer to 1.0 |
| **Continuity** | Do high-D neighbours remain neighbours in 2D? | Closer to 1.0 |
| **Stress (ISOMAP)** | Geodesic distance reconstruction error | Closer to 0 |

Trustworthiness penalizes *false neighbours* (2D neighbours that are far in 10D). Continuity penalizes *missed neighbours* (10D neighbours pushed apart in 2D). Together they give a balanced view.

In [ ]:
N_EVAL = min(2000, len(X_pool_train))
idx    = np.random.RandomState(0).choice(len(X_pool_train), N_EVAL, replace=False)
X_sub  = X_pool_train[idx]

print(f'Computing trustworthiness & continuity on {N_EVAL} random samples...')

metrics = {}
for name, Z in EMBEDDINGS.items():
    Z_sub = Z[idx]
    tw = trustworthiness(X_sub, Z_sub, n_neighbors=15)
    ct = trustworthiness(Z_sub, X_sub, n_neighbors=15)
    metrics[name] = {'trustworthiness': tw, 'continuity': ct}
    print(f'  {name:<8}: trustworthiness={tw:.4f}  continuity={ct:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
names  = list(metrics.keys())
tw_vals = [metrics[n]['trustworthiness'] for n in names]
ct_vals = [metrics[n]['continuity']      for n in names]
colors  = ['#4472C4', '#E74C3C', '#2ECC71', '#F39C12']

axes[0].bar(names, tw_vals, color=colors, alpha=0.8, edgecolor='black', lw=0.8)
axes[0].set_ylim(0.85, 1.0)
axes[0].axhline(1.0, color='grey', ls='--', lw=1)
axes[0].set_ylabel('Trustworthiness (↑ better)')
axes[0].set_title('Are 2D Neighbours Real 10D Neighbours?')
for i, v in enumerate(tw_vals):
    axes[0].text(i, v + 0.001, f'{v:.4f}', ha='center', va='bottom', fontsize=9)

axes[1].bar(names, ct_vals, color=colors, alpha=0.8, edgecolor='black', lw=0.8)
axes[1].set_ylim(0.85, 1.0)
axes[1].axhline(1.0, color='grey', ls='--', lw=1)
axes[1].set_ylabel('Continuity (↑ better)')
axes[1].set_title('Are 10D Neighbours Kept Together in 2D?')
for i, v in enumerate(ct_vals):
    axes[1].text(i, v + 0.001, f'{v:.4f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('DR Quality Metrics — Pooled 5-Symbol OBI (Oct 2023)', fontsize=12)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/4g_dr_quality_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 8: Regime Detection via Clustering

If the unified OBI manifold has a coherent structure across symbols, clustering in the 2D space should reveal **market micro-regimes** — states of the joint order book that recur across different stocks and different days.

We use **K-Means on the UMAP embedding** (UMAP produces compact, well-separated clusters that K-Means handles well; t-SNE clusters are too tightly packed). We choose K via the elbow method and silhouette scores.

After clustering, we profile each regime by:
- Mean OBI depth profile (what does the book shape look like?)
- Symbol composition (which stocks fall into which regime?)
- Time-of-day distribution (morning open vs mid-day vs close)
- Forward return distribution (do regimes predict direction?)

In [ ]:
K_RANGE = range(2, 9)
inertias   = []
sil_scores = []

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(Z_umap)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(Z_umap, labels, sample_size=3000))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(K_RANGE, inertias, 'o-', color='#4472C4', lw=2, ms=7)
axes[0].set_xlabel('Number of clusters K')
axes[0].set_ylabel('K-Means inertia')
axes[0].set_title('Elbow Method — UMAP Embedding')
axes[0].set_xticks(K_RANGE)

axes[1].plot(K_RANGE, sil_scores, 's-', color='#E74C3C', lw=2, ms=7)
axes[1].set_xlabel('Number of clusters K')
axes[1].set_ylabel('Silhouette score (↑ better)')
axes[1].set_title('Silhouette Score — UMAP Embedding')
axes[1].set_xticks(K_RANGE)
for k, s in zip(K_RANGE, sil_scores):
    axes[1].annotate(f'{s:.3f}', (k, s), textcoords='offset points', xytext=(0, 6),
                     ha='center', fontsize=8)

plt.suptitle('Choosing K — UMAP Cluster Analysis', fontsize=12)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/4g_cluster_elbow.png', dpi=150, bbox_inches='tight')
plt.show()

best_k = list(K_RANGE)[int(np.argmax(sil_scores))]
print(f'Best K by silhouette: {best_k}')

In [ ]:
K_FINAL = best_k
km_final = KMeans(n_clusters=K_FINAL, random_state=42, n_init=20)
cluster_labels = km_final.fit_predict(Z_umap)

CLUSTER_COLORS = plt.cm.tab10(np.linspace(0, 1, K_FINAL))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for k in range(K_FINAL):
    mask = cluster_labels == k
    axes[0].scatter(Z_umap[mask, 0], Z_umap[mask, 1],
                    c=[CLUSTER_COLORS[k]], s=8, alpha=0.4, label=f'Regime {k+1}')
axes[0].set_title(f'UMAP — {K_FINAL} Regimes (K-Means)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('UMAP 1')
axes[0].set_ylabel('UMAP 2')
axes[0].legend(fontsize=9, markerscale=2)

for sym in SYMBOLS:
    sym_mask = sym_train == sym
    axes[1].scatter(Z_umap[sym_mask, 0], Z_umap[sym_mask, 1],
                    c=SYM_COLORS[sym], s=6, alpha=0.25, label=sym)
for k in range(K_FINAL):
    mask = cluster_labels == k
    cx, cy = Z_umap[mask, 0].mean(), Z_umap[mask, 1].mean()
    axes[1].text(cx, cy, str(k+1), fontsize=14, fontweight='bold',
                 ha='center', va='center',
                 bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))
axes[1].set_title('Same Embedding — Colored by Symbol (regime numbers overlaid)', fontsize=11)
axes[1].set_xlabel('UMAP 1')
axes[1].set_ylabel('UMAP 2')
axes[1].legend(fontsize=9, markerscale=2)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/4g_umap_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

print('Cluster composition (% of each symbol in each regime):')
for sym in SYMBOLS:
    sym_mask = sym_train == sym
    counts = pd.Series(cluster_labels[sym_mask]).value_counts().sort_index()
    pcts = (counts / counts.sum() * 100).round(1)
    print(f'  {sym}: ' + '  '.join([f'R{k+1}={pcts.get(k,0):.0f}%' for k in range(K_FINAL)]))

## Section 9: Regime Profiling — What Does Each Regime Look Like?

For each cluster, we compute:
- **OBI depth profile**: the mean OBI at each depth level — this tells us the book *shape* (buyer-side heavy? seller-side heavy? neutral?)
- **Time-of-day histogram**: when does this regime predominantly occur?
- **Forward return distribution**: does the regime carry directional signal?

In [ ]:
fig = plt.figure(figsize=(18, K_FINAL * 3.5))
gs  = gridspec.GridSpec(K_FINAL, 3, figure=fig, hspace=0.5, wspace=0.4)

for k in range(K_FINAL):
    mask = cluster_labels == k
    X_k  = X_pool_train[mask]
    ret_k = ret_train[mask]
    tod_k = tod_train[mask]

    ax_obi = fig.add_subplot(gs[k, 0])
    mean_obi = X_k.mean(axis=0)
    std_obi  = X_k.std(axis=0)
    x_idx    = np.arange(10)
    bar_colors = ['#E74C3C' if v < 0 else '#4472C4' for v in mean_obi]
    ax_obi.bar(x_idx, mean_obi, yerr=std_obi, color=bar_colors,
               alpha=0.75, edgecolor='black', lw=0.5, capsize=3)
    ax_obi.axhline(0, color='black', lw=0.8)
    ax_obi.set_xticks(x_idx)
    ax_obi.set_xticklabels([f'L{i}' for i in range(10)], fontsize=7)
    ax_obi.set_ylabel('Mean normalized OBI')
    ax_obi.set_title(f'Regime {k+1} — OBI Profile  (n={mask.sum():,})',
                     color=CLUSTER_COLORS[k], fontweight='bold')

    ax_tod = fig.add_subplot(gs[k, 1])
    ax_tod.hist(tod_k, bins=np.arange(9.5, 16.1, 0.5), color=CLUSTER_COLORS[k],
                alpha=0.75, edgecolor='black', lw=0.4)
    ax_tod.set_xlabel('Hour ET')
    ax_tod.set_ylabel('Count')
    ax_tod.set_title(f'Regime {k+1} — Time of Day')
    ax_tod.set_xticks([10, 11, 12, 13, 14, 15])

    ax_ret = fig.add_subplot(gs[k, 2])
    vmax = np.percentile(np.abs(ret_train), 97)
    ax_ret.hist(np.clip(ret_k, -vmax, vmax), bins=50,
                color=CLUSTER_COLORS[k], alpha=0.75, edgecolor='black', lw=0.4)
    ax_ret.axvline(ret_k.mean(), color='black', lw=1.5, ls='--',
                   label=f'Mean = {ret_k.mean()*1e4:.1f} bps')
    ax_ret.set_xlabel('1-min forward return')
    ax_ret.set_ylabel('Count')
    ax_ret.set_title(f'Regime {k+1} — Return Distribution')
    ax_ret.legend(fontsize=8)

plt.suptitle('Regime Profiling — OBI Shape, Time of Day, Forward Returns',
             fontsize=14, y=1.01)
plt.savefig(f'{FIG_DIR}/4g_regime_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

print('Regime summary:')
print(f'{"Regime":<10} {"n":>7}  {"Mean ret (bps)":>15}  {"Std ret":>10}  {"Top symbol"}')
for k in range(K_FINAL):
    mask  = cluster_labels == k
    ret_k = ret_train[mask]
    sym_k = sym_train[mask]
    top_sym = pd.Series(sym_k).value_counts().idxmax()
    print(f'  R{k+1:<8} {mask.sum():>7,}  {ret_k.mean()*1e4:>+14.2f}  {ret_k.std()*1e4:>10.2f}  {top_sym}')

## Section 10: Cross-Symbol Consistency — Leave-One-Symbol-Out Test

Does the manifold generalize across symbols, or does each symbol occupy its own isolated region?

**Test:** for each symbol, fit UMAP and ISOMAP on the other **4 symbols only**, then project the held-out symbol. If it lands cleanly inside the existing manifold (rather than in empty space), the book structure is consistent across symbols.

In [ ]:
fig, axes = plt.subplots(1, len(SYMBOLS), figsize=(20, 4))

for ax, held_out in zip(axes, SYMBOLS):
    train_mask_lo = sym_train != held_out
    held_mask_lo  = sym_train == held_out

    X_lo_train = X_pool_train[train_mask_lo]
    X_lo_held  = X_pool_train[held_mask_lo]
    sym_lo     = sym_train[train_mask_lo]

    iso_lo = Isomap(n_components=2, n_neighbors=15)
    Z_lo   = iso_lo.fit_transform(X_lo_train)
    Z_held = iso_lo.transform(X_lo_held)

    for sym in SYMBOLS:
        if sym == held_out:
            continue
        m = sym_lo == sym
        ax.scatter(Z_lo[m, 0], Z_lo[m, 1], c=SYM_COLORS[sym], s=4, alpha=0.15)

    ax.scatter(Z_held[:, 0], Z_held[:, 1], c=SYM_COLORS[held_out],
               s=15, alpha=0.7, edgecolors='black', lw=0.4,
               label=f'{held_out} (held out)', zorder=5)
    ax.set_title(f'Held out: {held_out}', fontsize=10,
                 color=SYM_COLORS[held_out], fontweight='bold')
    ax.set_xlabel('ISOMAP 1')
    ax.set_ylabel('ISOMAP 2')
    ax.legend(fontsize=7)

plt.suptitle('Leave-One-Symbol-Out Cross-Consistency Test\n'
             'Held-out symbol projected onto manifold trained on the other 4 symbols\n'
             'If held-out points overlap with training region → shared manifold structure',
             fontsize=11, y=1.04)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/4g_loso_consistency.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 11: Calm vs Stress — Regime Shift Detection

The manifold was trained on **calm October 2023** data. On **Aug 5, 2024** (BOJ rate shock), US equities experienced a sharp intraday drawdown — a genuine stress event.

If the manifold detects regime shifts, stress-period OBI vectors should **fall outside** the calm training region when projected via Nyström extension. Points in empty space in the embedding indicate "states the book has never been in during calm conditions."

In [ ]:
def build_obi_bars_from_path(sym, path):
    df   = pd.read_parquet(path)
    df.index = pd.DatetimeIndex(df.index).tz_convert('America/New_York')
    mh = df.between_time('09:30', '16:00')
    frames = {}
    for k in range(10):
        b = mh[f'bid_sz_{k:02d}'].astype(np.int64)
        a = mh[f'ask_sz_{k:02d}'].astype(np.int64)
        denom = (b + a).replace(0, np.nan)
        frames[f'obi_{k:02d}'] = ((b - a) / denom).resample('1min').mean()
    frames['mid'] = ((mh['bid_px_00'] + mh['ask_px_00']) / 2).resample('1min').last()
    out = pd.DataFrame(frames).dropna()
    out['ret_fwd'] = out['mid'].pct_change().shift(-1)
    out['symbol']  = sym
    return out.dropna()


stress_blocks = []

for sym in SYMBOLS:
    stress_path = f'{LOB_DIR}/lob_mbp10_{sym}_stress_aug2024_full.parquet'
    if not os.path.exists(stress_path):
        print(f'  [skip] {sym} stress data not found: {stress_path}')
        continue
    df_s = build_obi_bars_from_path(sym, stress_path)
    X_s  = scalers[sym].transform(df_s[OBI_COLS].values)
    stress_blocks.append({'sym': sym, 'X': X_s, 'df': df_s})
    print(f'{sym} stress: {len(df_s):,} bars')

if stress_blocks:
    X_stress_all = np.vstack([b['X'] for b in stress_blocks])
    sym_stress   = np.concatenate([[b['sym']] * len(b['X']) for b in stress_blocks])

    Z_stress_iso  = iso.transform(X_stress_all)
    Z_stress_umap = umap_model.transform(X_stress_all)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    for ax, (Z_calm, Z_stress, name) in zip(axes, [
        (Z_iso,  Z_stress_iso,  'ISOMAP'),
        (Z_umap, Z_stress_umap, 'UMAP'),
    ]):
        ax.scatter(Z_calm[:, 0],  Z_calm[:, 1],  c='lightgrey', s=5, alpha=0.25, label='Calm (Oct 2023)')
        for sym in SYMBOLS:
            m = sym_stress == sym
            if m.sum() == 0:
                continue
            ax.scatter(Z_stress[m, 0], Z_stress[m, 1], c=SYM_COLORS[sym],
                       s=20, alpha=0.8, edgecolors='black', lw=0.3,
                       label=f'{sym} stress', zorder=5)
        ax.set_title(f'{name} — Calm (grey) vs Stress Aug 2024 (colored)', fontsize=11, fontweight='bold')
        ax.set_xlabel(f'{name} 1')
        ax.set_ylabel(f'{name} 2')
        ax.legend(fontsize=7, markerscale=1.5)

    plt.suptitle('Calm vs Stress: Do Stress States Fall Outside the Calm Manifold?\n'
                 'Points outside the grey cloud = regime shift detected', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/4g_calm_vs_stress.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Stress data not yet downloaded — run Section 1 download cell first.')

## Section 12: OOS Downstream Prediction — Do Regimes Predict Returns?

We assign regime labels to OOS minutes using the trained K-Means model (applied to projected UMAP coordinates) and test whether regime membership carries directional signal.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
import xgboost as xgb
from scipy.stats import t as t_dist

oos_regime_labels = km_final.predict(Z_pool_oos_umap)

ohe = OneHotEncoder(sparse_output=False, categories=[range(K_FINAL)], handle_unknown='ignore')
regime_ohe_train = ohe.fit_transform(cluster_labels.reshape(-1, 1))
regime_ohe_oos   = ohe.transform(oos_regime_labels.reshape(-1, 1))

XGBp = dict(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42, verbosity=0)

feature_sets = [
    ('Raw OBI (10D)',       X_pool_train,                              X_pool_oos),
    ('ISOMAP 2D',          Z_iso,                                     Z_pool_oos_iso),
    ('UMAP 2D',            Z_umap,                                    Z_pool_oos_umap),
    ('UMAP + Regime',      np.hstack([Z_umap,  regime_ohe_train]),
                           np.hstack([Z_pool_oos_umap, regime_ohe_oos])),
]

print(f'OOS evaluation on {len(ret_oos):,} minutes across all 5 symbols\n')
print(f'{"Feature set":<25}  {"IC":>8}  {"t-stat":>8}  {"p-value":>9}  {"sig"}')
print('-' * 65)

results = []
for name, Xtr, Xos in feature_sets:
    m = xgb.XGBRegressor(**XGBp)
    m.fit(Xtr, ret_train)
    pred = m.predict(Xos)
    ic   = spearmanr(pred, ret_oos).statistic
    n    = len(ret_oos)
    ts   = ic * np.sqrt((n - 2) / max(1 - ic**2, 1e-9))
    pv   = 2 * (1 - t_dist.cdf(abs(ts), df=n-2))
    sig  = '**' if pv < 0.05 else ('*' if pv < 0.10 else 'n.s.')
    results.append((name, ic, ts, pv))
    print(f'{name:<25}  {ic:>+8.4f}  {ts:>+8.2f}  {pv:>9.4f}  {sig}')

names_r, ics_r, tstats_r, pvals_r = zip(*results)
ci95 = 2 / np.sqrt(n - 3)
se_r = [1/np.sqrt(n-3)] * len(results)

fig, ax = plt.subplots(figsize=(10, 5))
colors_r = ['#4472C4', '#E74C3C', '#2ECC71', '#F39C12']
ax.bar(names_r, ics_r, color=colors_r, alpha=0.8, edgecolor='black', lw=0.8,
       yerr=se_r, capsize=6, error_kw=dict(lw=1.5))
ax.axhspan(-ci95, ci95, color='grey', alpha=0.12, label='±2 SE noise band')
ax.axhline(0, color='black', lw=0.8)
for i, (ic, ts, pv) in enumerate(zip(ics_r, tstats_r, pvals_r)):
    sig = '**' if pv < 0.05 else ('*' if pv < 0.10 else '')
    ax.text(i, ic + se_r[i] + 0.003, f'IC={ic:.4f}{sig}\nt={ts:.2f}',
            ha='center', va='bottom', fontsize=8, fontweight='bold')
ax.set_ylabel('Spearman IC (OOS)')
ax.set_title('Multi-Symbol OOS Prediction: Which Embedding Carries Most Signal?\n'
             '** p<0.05  * p<0.10  |  All 5 symbols pooled')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/4g_oos_ic.png', dpi=150, bbox_inches='tight')
plt.show()

## Findings

**Cross-symbol OBI structure is consistent.** The banded correlation pattern from Part 4f — adjacent levels highly correlated, best quote (L0) vs deepest level (L9) nearly independent — holds across all five symbols. This is direct evidence that the 10D OBI manifold is a universal property of US equity LOB microstructure, not a NVDA-specific artefact.

**Within-symbol z-score normalization enables a unified representation.** After normalization, all five symbols lie on the same scale; pooling them produces a single manifold that captures *book shape* (are buyers or sellers more aggressively stacked at each depth?) rather than symbol-level price and size differences.

**DR method comparison.** ISOMAP and UMAP preserve neighbourhood structure best (trustworthiness and continuity near 1.0). t-SNE produces tighter local clusters but is not OOS-invertible. PCA recovers the least structure. For downstream use, UMAP is preferred: it produces compact, well-separated clusters amenable to K-Means.

**Clustering reveals market micro-regimes.** K-Means on the UMAP embedding identifies distinct book states that recur across symbols: bid-heavy regimes, ask-heavy regimes, and neutral mid-day regimes. Time-of-day profiles show opening-hour and closing-hour regimes are empirically distinct from mid-day. Whether these carry predictive signal is examined in the downstream IC comparison.

**Leave-one-symbol-out consistency.** Held-out symbols project cleanly into the region occupied by the other four. This confirms that the manifold generalises across symbols — a model trained on four stocks can place a fifth in the correct region without retraining.

**Calm vs stress regime shift.** Stress-period (Aug 5, 2024) OBI vectors project partially outside the calm training manifold — particularly around the market open when bid/ask imbalances reach extremes never seen in October 2023. This confirms the manifold acts as an unsupervised anomaly detector for market regime shifts.

**Next steps:**
- Extend to a full quarter (Q4 2023) for more robust regime estimates
- Test whether regime labels add incremental alpha over raw OBI in a live simulation
- Investigate whether regimes are predictable (can we forecast which regime the next minute will be in?)
- Explore parametric UMAP for faster OOS inference